# N01 - Rebuild the cohort from the original KR file (plan stage 1)
Gate: sample flow reconciled; valid-HAZ rules and predictor codes signed off by **you**.

Plan rules followed here: stunted = `hw70 < -200` among valid HAZ (plan 3.1); special codes are never treated as ordinary values; every exclusion is counted with its rule (plan 3.2); raw `hw70` and the exclusion reason go to a **private** audit file; the KR file is read-only.

Row-level files go to `private/`. Only aggregate tables go to `outputs/`.

In [ ]:
# --- setup (same in every notebook) ---
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not on Colab - using local folder.")
import os, sys, time
import numpy as np
import pandas as pd
BASE_DIR = os.environ.get("THESIS_BASE_DIR", "/content/drive/MyDrive/THESIS_V2")
sys.path.append(BASE_DIR + "/code")
import bdhs_helpers as h
h.make_all_folders()
h.set_all_seeds(h.SEED_SPLIT)
h.print_environment("N01")

In [ ]:
# Read the original KR file (all columns) and keep only the columns this study needs
ID_COLS = ["caseid", "v001", "v002", "v003", "v005", "v021", "v022", "v023"]
PREDICTOR_COLS = ["b19", "b4", "bord", "b11", "v012", "v106", "v445", "v190", "v025", "v024", "v113", "v116", "v701", "v705"]
AUDIT_COLS = ["v135", "v501", "m14", "m15", "b5", "v438"]     # audit only - never predictors
OUTCOME_COLS = ["hw70"]
wanted = ID_COLS + PREDICTOR_COLS + AUDIT_COLS + OUTCOME_COLS

kr_all = pd.read_stata(h.KR_FILE, convert_categoricals=False)
print("KR file shape (rows, columns):", kr_all.shape)

present = []
missing_required = []
for col in wanted:
    if col in kr_all.columns:
        present.append(col)
    elif col in AUDIT_COLS:
        print("audit column not in file (ok, noted):", col)
    else:
        missing_required.append(col)
if len(missing_required) > 0:
    raise ValueError("Required columns missing from KR file: " + str(missing_required))

kr = kr_all[present].copy()
kr["row_id"] = np.arange(len(kr))     # stable key = row position in the original KR file
del kr_all
print("working table:", kr.shape)

### Value labels (code -> label) for the variables whose coding is still open
Read these tables yourself: this is where you decide what code 97, 98, 8, 9996-9999 mean.

In [ ]:
LABEL_VARS = ["b4", "v106", "v113", "v116", "v135", "v190", "v701", "v705", "v501", "v024", "v025"]
label_vars_present = []
for col in LABEL_VARS:
    if col in kr.columns:
        label_vars_present.append(col)
try:
    labelled = pd.read_stata(h.KR_FILE, columns=label_vars_present, convert_categoricals=True)
    for col in label_vars_present:
        combo = pd.concat([kr[col], labelled[col].astype(str)], axis=1)
        combo.columns = ["code", "label"]
        print("=====", col)
        print(combo.value_counts(dropna=False).sort_index().to_string())
except Exception as error:
    print("Could not read value labels:", error)
    print("Check the labels in the DHS recode manual / .DCT file instead.")

### Sample flow (T1) and exclusion reasons
Order of rules: age 0-59 -> has an anthropometry record -> (mother interviewed) -> DHS special HAZ code -> HAZ outside +-6 SD -> PSU missing.

In [ ]:
hw70 = kr["hw70"].values
age = kr["b19"].values
psu = kr["v021"].values

reasons = []
for i in range(len(kr)):
    if np.isnan(age[i]) or age[i] < 0 or age[i] > 59:
        reasons.append("1_age_outside_0_59")
    elif np.isnan(hw70[i]):
        reasons.append("2_no_anthropometry_record")
    elif hw70[i] >= 9990:
        reasons.append("3_special_code_" + str(int(hw70[i])))
    elif hw70[i] < -600 or hw70[i] > 600:
        reasons.append("4_haz_outside_6SD")
    elif np.isnan(psu[i]):
        reasons.append("5_psu_missing")
    else:
        reasons.append("included")
kr["exclusion_reason"] = reasons


def count_reason(prefix):
    n = 0
    for r in reasons:
        if r.startswith(prefix):
            n = n + 1
    return n


remaining = len(kr)
flow = []
flow.append(["0 all KR records", "none", 0, remaining])

removed = count_reason("1_")
remaining = remaining - removed
flow.append(["1 age 0-59 months", "keep 0 <= b19 <= 59", removed, remaining])

removed = count_reason("2_")
remaining = remaining - removed
flow.append(["2 eligible: has anthropometry record", "keep hw70 not missing", removed, remaining])

flow.append(["3 mother interviewed", "no rows removed: KR holds children of interviewed women only (DHS design - VERIFY)", 0, remaining])

removed = count_reason("3_")
remaining = remaining - removed
flow.append(["4 valid HAZ (special codes)", "drop hw70 >= 9990 (9996-9999 are DHS flags)", removed, remaining])

removed = count_reason("4_")
remaining = remaining - removed
flow.append(["5 plausible HAZ", "drop |HAZ| > 6 SD (hw70 beyond +-600)", removed, remaining])

removed = count_reason("5_")
remaining = remaining - removed
flow.append(["6 PSU present", "drop rows with missing v021", removed, remaining])

flow_df = pd.DataFrame(flow, columns=["step", "rule", "n_removed", "n_remaining"])
print(flow_df.to_string(index=False))
print()
print("Reference figures quoted in plan v2.2 (BDHS 2022 report): 4,420 eligible for anthropometry;")
print("4,199 with mother interviewed; supplied extract had 4,105 rows (provisional).")
print("Your rebuilt final cohort:", remaining, "rows")
print("Note: report percentages are survey-weighted, so an exact match of counts is not guaranteed.")

print()
print("Special hw70 codes present (counted over all KR rows):")
for code in [9996, 9997, 9998, 9999]:
    print("  ", code, ":", int((hw70 == code).sum()))

In [ ]:
# Build the cohort + outcome
cohort = kr[kr["exclusion_reason"] == "included"].copy()
cohort["stunted"] = (cohort["hw70"] < h.STUNTING_CUTOFF).astype(int)
print("cohort rows:", len(cohort), "| stunted:", int(cohort["stunted"].sum()),
      "| prevalence (unweighted):", round(cohort["stunted"].mean(), 4))
print("HAZ range in cohort (hundredths):", cohort["hw70"].min(), "to", cohort["hw70"].max())

# Survey design IDs (plan 3.4): are v001 == v021 and v022 == v023?
print()
print("v001 equals v021 in all rows:", bool((cohort["v001"] == cohort["v021"]).all()))
print("v022 equals v023 in all rows:", bool((cohort["v022"] == cohort["v023"]).all()))
print("Plan rule: use v021 as PSU and v022 as stratum.")
print("PSUs:", cohort["v021"].nunique(), "| strata:", cohort["v022"].nunique())

### U2 checks: the specific data issues listed in plan section 3.4
These tables are what you need to decide the coding. Nothing is changed here.

In [ ]:
key_counts = []

# b11: missing vs birth order
b11_missing = cohort["b11"].isna()
first_birth = cohort["bord"] == 1
n_first = int((b11_missing & first_birth).sum())
n_later = int((b11_missing & ~first_birth).sum())
print("b11 missing and bord == 1:", n_first, "| b11 missing and bord > 1:", n_later, "(plan saw 1,550 / 11 in the extract)")
print("b11 non-missing range (months expected):", cohort["b11"].min(), "to", cohort["b11"].max())
key_counts.append(["b11_missing_bord1", n_first])
key_counts.append(["b11_missing_bord_gt1", n_later])
if n_later > 0:
    print("Rows with b11 missing and bord > 1 (private row ids):")
    print(cohort[b11_missing & ~first_birth][["row_id", "bord", "b19"]].to_string(index=False))

# v113 / v116 code 97 vs residence status v135
print()
for var in ["v113", "v116"]:
    is97 = cohort[var] == 97
    print(var, "code 97 count:", int(is97.sum()))
    if "v135" in cohort.columns:
        print(pd.crosstab(is97, cohort["v135"], rownames=[var + "==97"], colnames=["v135"]).to_string())
    key_counts.append([var + "_code97", int(is97.sum())])

# v701 / v705 missing vs marital status v501
print()
for var in ["v701", "v705"]:
    miss = cohort[var].isna()
    print(var, "missing:", int(miss.sum()))
    key_counts.append([var + "_missing", int(miss.sum())])
    if "v501" in cohort.columns:
        print(pd.crosstab(miss, cohort["v501"], rownames=[var + " missing"], colnames=["v501"]).to_string())
print("v701 and v705 missing on the same rows:", bool((cohort["v701"].isna() == cohort["v705"].isna()).all()))

# v445 scale and special code
print()
v445_flag = int((cohort["v445"] >= 9990).sum())
v445_missing = int(cohort["v445"].isna().sum())
valid_bmi = cohort["v445"][cohort["v445"] < 9990] / 100.0
print("v445 flagged (>= 9990):", v445_flag, "| NaN:", v445_missing)
print("BMI after /100 (should look like 12-45):")
print(valid_bmi.describe().round(2).to_string())
key_counts.append(["v445_flagged", v445_flag])

# m14 / m15 are NOT predictors; just show missingness vs child age (plan 3.4)
if "m14" in cohort.columns:
    print()
    print("m14 missing by child age band (audit only):")
    band = pd.cut(cohort["b19"], [-1, 11, 23, 35, 47, 59])
    print(cohort["m14"].isna().groupby(band, observed=False).mean().round(3).to_string())

In [ ]:
# Save: sample flow + key counts (aggregate, outputs/); cohort + exclusion audit (row-level, private/)
h.save_table(flow_df, h.OUTPUT_DIR + "/T1_sample_flow.csv")
h.save_table(pd.DataFrame(key_counts, columns=["check", "count"]), h.OUTPUT_DIR + "/n01_key_counts.csv")

h.save_table(cohort, h.PRIVATE_DIR + "/cohort_private.csv")
audit = kr[["row_id", "caseid", "hw70", "exclusion_reason"]]
h.save_table(audit, h.PRIVATE_DIR + "/exclusion_audit_private.csv")
print("REMINDER: files in private/ contain survey identifiers. Do not upload or share them.")

### Gate - sign-off (you decide, not the code)
Tick each item in your head before setting `SIGN_OFF = True`:
- [ ] Sample flow reconciled or the difference explained (U1).
- [ ] You read the value labels and agree the hw70 special codes are handled correctly (D1).
- [ ] b11: the 11 missing cases with bord > 1 were looked at (U2). The code keeps them as `missing`, separate from `first_birth_NA`.
- [ ] v113/v116 code 97 and v701/v705 missing were looked at (U2). The code keeps them as their own levels - it does NOT merge them with other categories.
- [ ] v445 scale/flag (9998) confirmed.
- [ ] v021 is the PSU and v022 the stratum.
- [ ] **b11 binning (D14):** b11 is months, binned into fixed <24 / 24-35 / 36-47 / 48+ plus `first_birth_NA` and `missing`. This is my implementation choice, not in the plan text - approve or change it.

Any change you make to a rule must first be written into the plan's change log (plan page 1 rule).

In [ ]:
SIGN_OFF = False      # change to True ONLY after you checked the list above
if SIGN_OFF:
    f = open(h.OUTPUT_DIR + "/n01_signoff.txt", "w")
    f.write("N01 signed off. cohort rows = " + str(len(cohort)) + "\n")
    f.close()
    print("Gate passed. N02 may run.")
else:
    print("Gate NOT passed yet. N02 and N03 will refuse to run.")